# Camera-Radar BEV Fusion — Treinamento no Google Colab

Este notebook executa o treinamento do modelo **BEVFusionDetector** com fusão Câmera + Radar em visão BEV (Bird's Eye View) utilizando GPU T4.

### Pipeline de Execução:
1. **Verificar GPU (T4)**
2. **Conectar ao Google Drive** & Configurar caminho do repositório
3. **Instalar dependências** do projeto
4. **Baixar e descompactar nuScenes v1.0-mini** via `wget` no disco efêmero do Colab
5. **Validar estrutura das pastas** (sanity check dos diretórios `samples`, `v1.0-mini`, etc.)
6. **Treinar o modelo** (salvando checkpoints no Google Drive para persistência)
7. **Avaliar métricas** (Loss, IoU, Precision, Recall, F1-Score)
8. **Visualizar predições** (Câmera, Radar BEV, Ground Truth e Predição da Rede)

In [ ]:
# ─── Célula 1: Verificar GPU ───
import torch

print(f"PyTorch: {torch.__version__}")
print(f"CUDA disponível: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memória GPU Total: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("AVISO: GPU não detectada! Vá em 'Ambiente de Execução' -> 'Alterar tipo de ambiente de execução' e selecione T4 GPU.")

In [ ]:
# ─── Célula 2: Conectar ao Google Drive & Adicionar ao Python Path ───
import os
import sys
from google.colab import drive

# Montar o Google Drive
drive.mount('/content/drive')

# Caminho do repositório no seu Google Drive (ajuste se a sua pasta for diferente)
PROJECT_PATH = '/content/drive/MyDrive/PIBIC/fusion'

# Se o repositório estiver em outro local no Drive ou clonado diretamente em /content/fusion:
if not os.path.exists(PROJECT_PATH):
    if os.path.exists('/content/RC-fusion-architecture'):
        PROJECT_PATH = '/content/RC-fusion-architecture'
    elif os.path.exists('/content/fusion'):
        PROJECT_PATH = '/content/fusion'
    else:
        print(f"Pasta {PROJECT_PATH} não encontrada. Criando diretório ou verifique o caminho do Drive.")

if PROJECT_PATH not in sys.path:
    sys.path.insert(0, PROJECT_PATH)

print(f"Diretório do Projeto ativo: {PROJECT_PATH}")

In [ ]:
# ─── Célula 3: Instalar Dependências ───
req_path = os.path.join(PROJECT_PATH, 'requirements_colab.txt')
if os.path.exists(req_path):
    !pip install -q -r {req_path}
else:
    !pip install -q nuscenes-devkit einops pyyaml tqdm matplotlib tensorboard pycocotools

print("Dependências instaladas com sucesso!")

In [ ]:
# ─── Célula 4: Baixar nuScenes mini via wget e descompactar no disco rápido do Colab ───
import os

DATA_DIR = '/content/data/sets'
NUSCENES_PATH = os.path.join(DATA_DIR, 'nuscenes')
TAR_FILE = '/content/v1.0-mini.tgz'

os.makedirs(DATA_DIR, exist_ok=True)

# Verifica se os metadados já foram descompactados
if not os.path.exists(os.path.join(NUSCENES_PATH, 'v1.0-mini')):
    if not os.path.exists(TAR_FILE):
        print("Baixando nuScenes v1.0-mini via wget (~1.5 GB)... Aguarde.")
        !wget --progress=bar:force -O {TAR_FILE} https://www.nuscenes.org/data/v1.0-mini.tgz
    else:
        print("Arquivo compactado já existe no ambiente efêmero.")
        
    print("Extraindo arquivos para /content/data/sets/ ...")
    !tar -xzf {TAR_FILE} -C {DATA_DIR}/
    print("Extração concluída!")
else:
    print("nuScenes v1.0-mini já descompactado e disponível.")

In [ ]:
# ─── Célula 5: Validar a Estruturação da Pasta no Colab ───
import os

expected_subdirs = ['v1.0-mini', 'samples', 'sweeps', 'maps']
print(f"Validando estrutura de pastas em: {NUSCENES_PATH}")

all_ok = True
if not os.path.exists(NUSCENES_PATH):
    print(f"[ERRO] Diretório raiz do dataset não existe: {NUSCENES_PATH}")
    all_ok = False
else:
    for subdir in expected_subdirs:
        subdir_path = os.path.join(NUSCENES_PATH, subdir)
        if os.path.exists(subdir_path):
            count = len(os.listdir(subdir_path))
            print(f"  [OK] {subdir}/ encontrado ({count} itens/arquivos)")
        else:
            print(f"  [FALHA] Subdiretório obrigatório ausente: {subdir}/")
            all_ok = False

assert all_ok, "Estrutura do dataset incompleta! Verifique o processo de extração do arquivo .tgz."

# Teste rápido de instanciação com nuScenes devkit
from nuscenes.nuscenes import NuScenes
nusc = NuScenes(version='v1.0-mini', dataroot=NUSCENES_PATH, verbose=False)
print(f"[SUCESSO] nuScenes devkit inicializado com {len(nusc.scene)} cenas cadastradas.")

In [ ]:
# ─── Célula 6: Configurar Paths e Parâmetros de Treinamento ───
from src.engine.train import load_config

CONFIG_PATH = os.path.join(PROJECT_PATH, 'config', 'default.yaml')
cfg = load_config(CONFIG_PATH)

# Configurar caminhos para o ambiente Colab + Drive
cfg['data']['dataroot'] = NUSCENES_PATH
cfg['training']['checkpoint_dir'] = os.path.join(PROJECT_PATH, 'checkpoints')
cfg['training']['log_dir'] = os.path.join(PROJECT_PATH, 'logs')

os.makedirs(cfg['training']['checkpoint_dir'], exist_ok=True)
os.makedirs(cfg['training']['log_dir'], exist_ok=True)

print(f"Configuração carregada de: {CONFIG_PATH}")
print(f"Dataset dataroot: {cfg['data']['dataroot']}")
print(f"Checkpoints serão salvos no Drive em: {cfg['training']['checkpoint_dir']}")
print(f"Logs TensorBoard no Drive em: {cfg['training']['log_dir']}")

In [ ]:
# ─── Célula 7: Treinar o Modelo ───
from src.engine.train import train

# Executa o treinamento completo (50 épocas ou Early Stopping)
train(
    config_path=CONFIG_PATH,
    dataroot=NUSCENES_PATH,
)

In [ ]:
# ─── Célula 8: Avaliar Métricas do Melhor Checkpoint ───
import torch
from torch.utils.data import DataLoader
from src.engine.evaluate import evaluate, print_metrics
from src.engine.train import build_model, load_checkpoint
from src.dataset.nuscenes_dataset import NuScenesFusionDataset, collate_fn

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = build_model(cfg, device)

CKPT_PATH = os.path.join(cfg['training']['checkpoint_dir'], 'best_model.pth')

if os.path.exists(CKPT_PATH):
    epoch, metrics = load_checkpoint(model, None, CKPT_PATH, device)
    print(f"Melhor modelo carregado com sucesso da época {epoch}!")

    val_dataset = NuScenesFusionDataset(
        dataroot=NUSCENES_PATH,
        version='v1.0-mini',
        split='val',
        image_size=tuple(cfg['data']['image_size']),
        bev_size=tuple(cfg['data']['bev_size']),
        bev_range=tuple(cfg['data']['bev_range']),
        radar_max_points=cfg['data']['radar_max_points'],
    )
    val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False, collate_fn=collate_fn)
    results = evaluate(model, val_loader, device, cfg)
    print_metrics(results)
else:
    print(f"Checkpoint não encontrado em {CKPT_PATH}. Conclua o treino primeiro.")

In [ ]:
# ─── Célula 9: Visualização Comparativa de Predições (Câmera, Radar BEV, GT e Pred) ───
import matplotlib.pyplot as plt
import numpy as np
from src.dataset.radar_transforms import radar_to_bev_grid

model.eval()

# Selecionar uma amostra de teste/validação
sample_idx = 0
sample = val_dataset[sample_idx]

image = sample['image'].unsqueeze(0).to(device)
radar_points = sample['radar_points'].unsqueeze(0).to(device)
radar_mask = sample['radar_mask'].unsqueeze(0).to(device)

# Projeção Radar no Grid BEV
radar_bev = radar_to_bev_grid(
    radar_points[0], radar_mask[0],
    bev_size=tuple(cfg['data']['bev_size']),
    bev_range=tuple(cfg['data']['bev_range']),
).unsqueeze(0).to(device)

with torch.no_grad():
    seg_logits = model(image, radar_bev)
    seg_prob = torch.sigmoid(seg_logits).cpu().squeeze().numpy()

# Imagem Desnormalizada para exibição
img_show = sample['image'].permute(1, 2, 0).cpu().numpy()
img_show = (img_show * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406]))
img_show = np.clip(img_show, 0, 1)

# Visualização lado a lado (4 painéis)
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

# 1. Câmera Frontal
axes[0].imshow(img_show)
axes[0].set_title("Câmera Frontal (RGB)")
axes[0].axis('off')

# 2. Radar BEV (Canal 0: Densidade / Presença)
radar_density = radar_bev[0, 0].cpu().numpy()
axes[1].imshow(radar_density, cmap='viridis')
axes[1].set_title("Radar BEV (Densidade/Presença)")
axes[1].axis('off')

# 3. Ground Truth BEV
gt = sample['bev_segmentation'].squeeze().cpu().numpy()
axes[2].imshow(gt, cmap='magma', vmin=0, vmax=1)
axes[2].set_title("Ground Truth BEV")
axes[2].axis('off')

# 4. Predição da Rede Fusão
axes[3].imshow(seg_prob, cmap='magma', vmin=0, vmax=1)
axes[3].set_title(f"Predição Modelo (Probabilidade)")
axes[3].axis('off')

plt.tight_layout()
plt.show()